# ML-06 — Signal Audit: Do the Flags Hold?

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/SodiqAbdulwaris/flyrank-internship-ml/blob/main/work/notebooks/w04_signal_audit.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

**Optional stretch — retired 2026-07-13.** Its core already lives in ML-07's baseline reason codes and ML-08's error analysis. This notebook runs three quick, honest signal checks rather than duplicating that depth.

## 1. Distributions

*Look before deciding: distributions of your key fields. Note the heavy tails.*

Distributions of the two fields the baseline rule (ML-07) is built from.

In [1]:
import pandas as pd

df = pd.read_csv("data/raw/content_refresh_anonymized.csv")
df["is_declining_label"] = (df["trend_direction"] == "down").astype(int)

print("impressions_90d: mean", round(df['impressions_90d'].mean(), 1),
      "| median", df["impressions_90d"].median(), "-- heavy right tail")
print("days_since_last_update: mean", round(df['days_since_last_update'].mean(), 1),
      "| median", df["days_since_last_update"].median())


impressions_90d: mean 5200.4 | median 731.0 -- heavy right tail
days_since_last_update: mean 46.1 | median 20.0


## 2. Signal test #1 / #2 / #3 (verdict each)

*Three safe signals, each with a mini-test and a verdict: CONFIRMED / OPPOSITE / MIXED / FALSE.*

**Test 1 — declining pages have lower CTR.** **Test 2 — stale pages (>=180d) decline more often.** **Test 3 — high search-volume pages decline more often.**

In [2]:
# Test 1: CTR by declining status
ctr_by_label = df.groupby("is_declining_label")["ctr"].mean().round(3)
print("mean CTR, not-declining vs declining:", ctr_by_label.to_dict())
print("Verdict: CONFIRMED -- declining pages average 0.32% CTR vs 0.73% for stable/growing pages.\n")

# Test 2: staleness alone vs declining rate
stale = df["days_since_last_update"] >= 180
stale_rate = df.groupby(stale)["is_declining_label"].mean().round(3)
print("declining rate, not-stale vs stale (>=180d):", stale_rate.to_dict())
print("Verdict: OPPOSITE of the assumption -- staleness ALONE is not linked to a higher decline",
      "rate here (0.471 stale vs 0.542 not-stale). The baseline rule works only by ANDing",
      "staleness WITH visibility (see the flag-linked test below), not from staleness alone.\n")

# Test 3: search_volume vs declining rate
corr = round(df["search_volume"].corr(df["is_declining_label"]), 4)
sv_high = df["search_volume"] >= df["search_volume"].median()
sv_rate = df.groupby(sv_high)["is_declining_label"].mean().round(3)
print("corr(search_volume, is_declining_label):", corr)
print("declining rate, below- vs above-median search_volume:", sv_rate.to_dict())
print("Verdict: FALSE / negligible -- consistent with ML-02's finding that search_volume",
      "barely tracks real performance in this portfolio.")


mean CTR, not-declining vs declining: {0: 0.732, 1: 0.324}
Verdict: CONFIRMED -- declining pages average 0.32% CTR vs 0.73% for stable/growing pages.

declining rate, not-stale vs stale (>=180d): {False: 0.542, True: 0.471}
Verdict: OPPOSITE of the assumption -- staleness ALONE is not linked to a higher decline rate here (0.471 stale vs 0.542 not-stale). The baseline rule works only by ANDing staleness WITH visibility (see the flag-linked test below), not from staleness alone.

corr(search_volume, is_declining_label): -0.0191
declining rate, below- vs above-median search_volume: {False: 0.572, True: 0.518}
Verdict: FALSE / negligible -- consistent with ML-02's finding that search_volume barely tracks real performance in this portfolio.


## 3. The flag-linked test

*Pick a signal one of FlyRank's real flags relies on. Does the data support the rule's assumption?*

The `stale_but_visible` flag (ML-07's baseline, reused in ML-10's reason codes) ANDs staleness with visibility. Does that combination hold up better than staleness alone did above?

In [3]:
flag = (df["days_since_last_update"] >= 180) & (df["impressions_90d"] >= 500)
print("stale_but_visible: n =", int(flag.sum()))
print("declining rate among flagged pages:", round(df.loc[flag, "is_declining_label"].mean(), 3))
print("overall base rate:", round(df["is_declining_label"].mean(), 3))
print("\nVerdict: CONFIRMED, but n=17 is small enough that two fewer declining pages in this",
      "group would materially change the rate -- read as directional, not proof (matches",
      "ML-07's finding that this flag is precise but rare).")


stale_but_visible: n = 17
declining rate among flagged pages: 0.941
overall base rate: 0.542

Verdict: CONFIRMED, but n=17 is small enough that two fewer declining pages in this group would materially change the rate -- read as directional, not proof (matches ML-07's finding that this flag is precise but rare).


## 4. What this means in practice

*Two or three sentences: what a content team should take from this.*

A single-field rule ('this page is stale, flag it') doesn't hold up — staleness alone is not linked to decline in this portfolio. Only the AND-combination of staleness and visibility does, and even that rests on very few pages (17). CTR is the one field that cleanly separates declining from stable/growing pages on its own. A content team reading this: don't triage on "hasn't been updated in a while" by itself — pair it with a traffic-visibility check, and treat low CTR as the more reliable single-field warning sign.

In [4]:
# No further computation -- this section is the plain-words takeaway from the tests above.
print("mean CTR gap (not-declining minus declining):", round(ctr_by_label[0] - ctr_by_label[1], 3))


mean CTR gap (not-declining minus declining): 0.408


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.